### Looking at the data collected ###

In [1]:
import os
import pandas as pd
import numpy as np
import math


print(os.getcwd())

c:\Users\yinki\OneDrive\NUS\BT4101\fyp-kiat\ML_Core\data\raw_data


### Prices Daily ###

In [2]:
ohlc_data = pd.read_parquet(f'daily_ohlc_data.parquet')
ohlc_data[ohlc_data['Ticker'] == 'AAPL']

Price,Date,Open,High,Low,Close,Adj Close,Volume,Ticker
2,2005-01-03,1.156786,1.162679,1.117857,1.130179,0.949987,6.919920e+08,AAPL
872,2005-01-04,1.139107,1.169107,1.124464,1.141786,0.959743,1.096810e+09,AAPL
1742,2005-01-05,1.151071,1.165179,1.143750,1.151786,0.968149,6.804336e+08,AAPL
2612,2005-01-06,1.154821,1.159107,1.130893,1.152679,0.968900,7.055552e+08,AAPL
3482,2005-01-07,1.160714,1.243393,1.156250,1.236607,1.039446,2.227450e+09,AAPL
...,...,...,...,...,...,...,...,...
5083852,2025-09-09,237.000000,238.779999,233.360001,234.350006,234.350006,6.631390e+07,AAPL
5084852,2025-09-10,232.190002,232.419998,225.949997,226.789993,226.789993,8.344080e+07,AAPL
5085852,2025-09-11,226.880005,230.449997,226.649994,230.029999,230.029999,5.020860e+07,AAPL
5086852,2025-09-12,229.220001,234.509995,229.020004,234.070007,234.070007,5.582420e+07,AAPL


In [3]:
ticker_starts = {}
for ticker in ohlc_data.dropna()['Ticker'].unique():
    try:
        dates = ohlc_data[ohlc_data['Ticker'] == ticker].reset_index(drop=True)['Date']
        ticker_starts[ticker] = [dates.iloc[0],  dates.iloc[-1]]
    except Exception as e:
        print(ticker, str(e))


In [4]:
#ticker_starts = sorted([[k, v[0], v[1]] for k, v in ticker_starts.items()], key=lambda x: x[1], reverse=True)
tickers = [i[0] for i in ticker_starts]
start_date = '2014-08-08'
end_date = '2025-09-15'

In [55]:
trading_disclosure = pd.read_parquet(f'congressional_trading_disclosure.parquet')
earnings_quality_score = pd.read_parquet(f'earnings_quality_score.parquet')
earnings_surprises = pd.read_parquet(f'earnings_surprises.parquet')
employee_count = pd.read_parquet(f'employee_count_history.parquet')
bs_quarterly = pd.read_parquet(f'financials_bs_quarterly_all_tickers.parquet')
cf_quarterly = pd.read_parquet(f'financials_cf_quarterly_all_tickers.parquet')
ic_quarterly = pd.read_parquet(f'financials_ic_quarterly_all_tickers.parquet')
historical_esg = pd.read_parquet(f'historical_esg_scores.parquet')
insider_sentiment = pd.read_parquet(f'insider_sentiment.parquet')
insider_transactions = pd.read_parquet(f'insider_transactions.parquet')
market_cap = pd.read_parquet(f'market_cap_history.parquet')
social_sentiment = pd.read_parquet(f'social_sentiment.parquet')
usa_spending_history = pd.read_parquet(f'usa_spending_history.parquet')

### Treatment of Trading Disclosures ###

In [6]:
trading_disclosure[(trading_disclosure['amountFrom'] < trading_disclosure['amountTo']) & (trading_disclosure['transactionType'] == 'Sale')]
trading_disclosure[(trading_disclosure['amountFrom'] > trading_disclosure['amountTo']) & (trading_disclosure['transactionType'] == 'Purchase')]

,amountFrom,amountTo,assetName,filingDate,name,ownerType,position,symbol,transactionDate,transactionType


In [7]:
# We flip all the amountFrom to amountTo if there is any violating conditions
mask_sale = (trading_disclosure['transactionType'] == 'Sale') & \
            (trading_disclosure['amountFrom'] < trading_disclosure['amountTo'])

mask_purchase = (trading_disclosure['transactionType'] == 'Purchase') & \
                (trading_disclosure['amountFrom'] > trading_disclosure['amountTo'])

# Combine both masks
mask = mask_sale | mask_purchase

# Swap amountFrom and amountTo for violating rows
trading_disclosure.loc[mask, ['amountFrom', 'amountTo']] = \
    trading_disclosure.loc[mask, ['amountTo', 'amountFrom']].values


# Check for violation
print(len(trading_disclosure[(trading_disclosure['amountFrom'] < trading_disclosure['amountTo']) & (trading_disclosure['transactionType'] == 'Sale')]))
print(len(trading_disclosure[(trading_disclosure['amountFrom'] > trading_disclosure['amountTo']) & (trading_disclosure['transactionType'] == 'Purchase')]))

0
0


In [8]:
# Aggregate function
trading_disclosure_agg = (trading_disclosure.groupby(['transactionDate', 'symbol']).apply(lambda g: pd.Series({'totalAmountFrom': g['amountFrom'].sum(),'totalAmountTo': g['amountTo'].sum()})).reset_index()).sort_values(by=['transactionDate', 'symbol']).rename(columns={'transactionDate': 'Date', 'symbol': 'Ticker'})
trading_disclosure_agg

C:\Users\yinki\AppData\Local\Temp\ipykernel_33652\4035043030.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  trading_disclosure_agg = (trading_disclosure.groupby(['transactionDate', 'symbol']).apply(lambda g: pd.Series({'totalAmountFrom': g['amountFrom'].sum(),'totalAmountTo': g['amountTo'].sum()})).reset_index()).sort_values(by=['transactionDate', 'symbol']).rename(columns={'transactionDate': 'Date', 'symbol': 'Ticker'})


,Date,Ticker,totalAmountFrom,totalAmountTo
0,2013-01-24,AAPL,1001,15000
1,2013-04-19,AAPL,55055,825000
2,2013-04-29,CG,100001,250000
3,2013-06-15,NEE,2002,30000
4,2013-07-31,AAPL,1001,15000
...,...,...,...,...
2420,2025-08-21,ZION,15000,1000
2421,2025-08-22,CSL,15000,1000
2422,2025-08-26,MSCI,15000,1000
2423,2025-08-26,UNP,15001,50000


### Treatment of earnings quality score ###

In [9]:
# We dont need the letter score and freq, we will just left join and forward fill
earnings_quality_score_agg = earnings_quality_score[['period', 'symbol', 'cashGenerationCapitalAllocation', 'growth', 'leverage', 'profitability', 'score']].rename(columns={'period': 'Date', 'symbol': 'Ticker'}).sort_values(by=['Date', 'Ticker']).reset_index(drop=True)
earnings_quality_score_agg

,Date,Ticker,cashGenerationCapitalAllocation,growth,leverage,profitability,score
0,1989-03-31,DVN,10.399807,0.000000,0.000000,0.000000,2.599952
1,1989-06-30,AAPL,94.024450,84.285900,0.000000,41.786827,55.024296
2,1989-06-30,CBSH,37.265650,0.000000,0.000000,40.384070,25.883240
3,1989-06-30,CCK,0.000000,45.529175,3.346749,100.000000,37.218983
4,1989-06-30,CHCO,37.265650,0.000000,37.651714,37.733010,37.550125
...,...,...,...,...,...,...,...
28521,2025-06-30,ZWS,81.075096,82.398636,47.003925,40.454323,62.732994
28522,2025-07-31,WMT,35.486423,67.640420,52.290590,100.000000,63.854360
28523,2025-08-02,BBWI,27.655193,44.711520,52.424507,14.579963,34.842796
28524,2025-08-02,MRVL,73.558640,72.692070,42.832380,14.462291,50.886340


### Treatment of earning surprises ###

In [10]:
earnings_surprises_agg = earnings_surprises[['period', 'symbol', 'surprisePercent']].rename(columns={'period': 'Date', 'symbol': 'Ticker', 'surprisePercent':'surprise_percent_quarterly'}).sort_values(by=['Date', 'Ticker']).reset_index(drop=True)
earnings_surprises_agg

,Date,Ticker,surprise_percent_quarterly
0,1996-06-30,WETH,-1.9608
1,1996-06-30,WETH,-1.9608
2,1997-06-30,ACMT,-0.3577
3,1997-09-30,ACMT,-23.9045
4,1997-12-31,ACMT,-22.3301
...,...,...,...
2479,2025-09-30,ULTA,11.7804
2480,2025-09-30,VSCO,183.5052
2481,2025-09-30,WLYB,-2.9703
2482,2025-09-30,WMT,-9.1638


### Treatment of employee count history ###

In [11]:
employee_count_agg = employee_count.rename(columns={'atDate': 'Date', 'symbol': 'Ticker'}).sort_values(by=[ 'Ticker', 'Date']).reset_index(drop=True)
employee_count_agg


,Date,employee,Ticker
0,2005-09-24,14800,AAPL
1,2007-09-29,21600,AAPL
2,2008-09-27,32000,AAPL
3,2009-09-26,34300,AAPL
4,2010-09-25,46600,AAPL
...,...,...,...
652,2008-12-27,6800,ZWS
653,2009-06-27,5754,ZWS
654,2009-12-26,5650,ZWS
655,2011-07-02,6300,ZWS


### Treatment of financial statements ###

In [36]:
bs_quarterly

,accountsPayable,accountsReceivables,accruedLiability,accumulatedDepreciation,additionalPaidInCapital,cash,cashEquivalents,cashShortTermInvestments,commonStock,currentAssets,...,insurancePolicyLiabilities,insuranceReceivables,grossPremiumsEarned,preferredSharesOutstanding,otherRevenue,deferredPolicyAcquisitionCosts,benefitsClaimsLossAdjustment,operationsMaintenance,purchasedFuelPowerGas,deferredPolicyAcquisitionExpense
0,289.109,73.665,363.764,-2968.851,1720.04,988.461,13.710,1002.171,659.301,2791.920,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,253.221,23.199,384.322,-2902.346,1720.04,864.399,3.617,868.016,659.301,2326.294,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,266.834,271.490,343.010,-2843.659,1720.04,749.331,4.949,754.280,659.301,2332.417,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,259.709,324.081,378.944,-2781.843,1720.04,1209.220,56.606,1265.826,659.301,2730.248,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,253.636,204.187,290.811,-2717.020,1720.04,1325.722,7.308,1333.030,659.301,2695.485,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163019,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
163020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
163021,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
163022,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [37]:
ic_quarterly

,costOfGoodsSold,dilutedAverageSharesOutstanding,dilutedEPS,ebit,grossIncome,netIncome,netIncomeAfterTaxes,period,pretaxIncome,provisionforIncomeTaxes,researchDevelopment,revenue,sgaExpense,totalOperatingExpense,totalOtherIncomeExpenseNet,interestExpense,interestIncomeExpense,nonRecurringItems,symbol,statement_type
0,50318.000,14948.179,1.57,28202.000,43718.000,23434.000,23434.000,2025-06-28,28031.000,4597.000,8866.000,94036.000,6650.000,15516.000,-171.000,NaN,NaN,NaN,AAPL,ic
1,50492.000,15056.133,1.65,29589.000,44867.000,24780.000,24780.000,2025-03-29,29310.000,4530.000,8550.000,95359.000,6728.000,15278.000,-279.000,NaN,NaN,NaN,AAPL,ic
2,66025.000,15150.865,2.40,42832.000,58275.000,36330.000,36330.000,2024-12-28,42584.000,6254.000,8268.000,124300.000,7175.000,15443.000,-248.000,NaN,NaN,NaN,AAPL,ic
3,51051.000,15242.855,0.96,29591.000,43879.000,14736.000,24982.000,2024-09-28,29610.000,4628.000,7765.000,94930.000,6523.000,14288.000,19.000,NaN,NaN,NaN,AAPL,ic
4,46099.000,15348.175,1.40,25352.000,39678.000,21448.000,21448.000,2024-06-29,25494.000,4046.000,8006.000,85777.000,6320.000,14326.000,142.000,NaN,NaN,NaN,AAPL,ic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
140,628.188,13941.648,0.01,182.241,736.572,119.764,119.764,1990-06-29,196.335,76.571,124.171,1364.760,430.160,554.331,14.094,NaN,0.0,0.0,AAPL,ic
141,609.590,14219.408,0.01,199.172,736.612,131.815,131.815,1990-03-30,216.092,84.277,115.235,1346.202,422.205,537.440,16.920,NaN,0.0,0.0,AAPL,ic
142,716.210,14503.552,0.01,190.084,777.173,124.840,124.840,1989-12-29,204.656,79.816,106.311,1493.383,480.778,587.089,14.572,NaN,0.0,0.0,AAPL,ic
143,677.428,14549.136,0.01,171.648,706.319,161.080,161.080,1989-09-29,264.067,102.987,113.146,1383.747,421.525,534.671,92.419,NaN,0.0,0.0,AAPL,ic


In [54]:
cf_quarterly

,capex,cashDividendsPaid,cashTaxesPaid,changeinCash,changesinWorkingCapital,depreciationAmortization,fcf,issuanceReductionCapitalStock,issuanceReductionDebtNet,netCashFinancingActivities,...,otherFundsFinancingItems,otherFundsNonCashItems,otherInvestingCashFlowItemsTotal,period,stockBasedCompensation,cashInterestPaid,deferredTaxesInvestmentTaxCredit,foreignExchangeEffects,symbol,statement_type
0,-3462.000,-3945.000,5649.000,8107.000,-2034.000,2830.000,24405.000,-21075.000,2711.000,-24833.000,...,-2524.0,3637.000,8535.000,2025-06-28,3168.0,NaN,NaN,NaN,AAPL,cf
1,-3071.000,-3758.000,13032.000,-2137.000,-6507.000,2661.000,20881.000,-25898.000,976.000,-29006.000,...,-326.0,3018.000,5988.000,2025-03-29,3226.0,NaN,NaN,NaN,AAPL,cf
2,-2940.000,-3856.000,18651.000,356.000,-10752.000,3080.000,26995.000,-23606.000,-8953.000,-39371.000,...,-2956.0,1277.000,12732.000,2024-12-28,3286.0,NaN,NaN,NaN,AAPL,cf
3,-2908.000,-3804.000,6872.000,3308.000,6608.000,2911.000,23903.000,-25083.000,4387.000,-24948.000,...,-448.0,2556.000,4353.000,2024-09-28,2858.0,NaN,NaN,NaN,AAPL,cf
4,-2151.000,-3895.000,4699.000,-7286.000,1684.000,2850.000,26707.000,-26522.000,-3253.000,-36017.000,...,-2347.0,2876.000,2024.000,2024-06-29,2869.0,NaN,NaN,NaN,AAPL,cf
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
140,-50.763,-13.452,0.000,19.162,86.367,49.858,251.083,-152.536,4.924,-161.064,...,0.0,0.535,-70.857,1990-06-29,NaN,0.000,45.322,0.0,AAPL,cf
141,-42.823,-13.522,0.000,235.303,73.342,44.291,260.426,-90.772,-47.926,-152.220,...,0.0,0.163,127.097,1990-03-30,NaN,0.000,53.638,0.0,AAPL,cf
142,-62.868,-13.970,0.000,-27.792,54.071,47.753,198.265,-73.904,11.148,-76.726,...,0.0,0.538,-149.331,1989-12-29,NaN,0.000,33.931,0.0,AAPL,cf
143,-59.706,-13.092,170.755,272.223,-72.825,40.052,18.931,14.497,-9.857,-8.452,...,0.0,-78.398,261.744,1989-09-29,NaN,15.954,28.728,0.0,AAPL,cf


In [56]:
# Merge on period + symbol
# Merge iteratively on period + symbol
dfs = [bs_quarterly, ic_quarterly, cf_quarterly]
df_agg = ( bs_quarterly .merge(ic_quarterly, on=["period", "symbol"], suffixes=("_bs", "_ic")) .merge(cf_quarterly, on=["period", "symbol"], suffixes=("", "_cf")) )

In [57]:
df_agg

,accountsPayable,accountsReceivables,accruedLiability,accumulatedDepreciation,additionalPaidInCapital,cash,cashEquivalents,cashShortTermInvestments,commonStock,currentAssets,...,otherFundsFinancingItems,otherInvestingCashFlowItemsTotal,statement_type,cashDividendsPaid,changesinWorkingCapital,depreciationAmortization_cf,netIncomeStartingLine,otherFundsNonCashItems,deferredTaxesInvestmentTaxCredit,stockBasedCompensation
0,289.1090,73.6650,363.764,-2968.851,1720.0400,988.461,13.7100,1002.1710,659.3010,2791.9200,...,-36.881,NaN,cf,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,253.2210,23.1990,384.322,-2902.346,1720.0400,864.399,3.6170,868.0160,659.3010,2326.2940,...,-74.309,NaN,cf,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,266.8340,271.4900,343.010,-2843.659,1720.0400,749.331,4.9490,754.2800,659.3010,2332.4170,...,1.370,0.0,cf,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,259.7090,324.0810,378.944,-2781.843,1720.0400,1209.220,56.6060,1265.8260,659.3010,2730.2480,...,105.011,0.0,cf,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,253.6360,204.1870,290.811,-2717.020,1720.0400,1325.722,7.3080,1333.0300,659.3010,2695.4850,...,-11.898,NaN,cf,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50936,0.3589,0.0302,NaN,NaN,0.5797,NaN,2.5482,2.5482,13.5792,2.9055,...,NaN,0.0,cf,NaN,0.2230,0.1322,-0.7120,0.0913,NaN,NaN
50937,0.0397,0.0740,NaN,NaN,0.4908,NaN,2.8115,2.8115,13.5746,3.0710,...,NaN,0.0,cf,NaN,0.0803,0.1351,-0.7513,0.0873,NaN,NaN
50938,0.1058,0.0550,NaN,NaN,0.4035,NaN,3.2602,3.2602,13.5746,3.6749,...,NaN,0.0,cf,NaN,-0.1001,0.1351,-0.7374,0.1412,NaN,NaN
50939,0.0733,0.0409,NaN,NaN,0.2623,NaN,3.8213,3.8213,13.5746,4.1027,...,NaN,0.0,cf,NaN,-0.0172,0.1337,-0.8351,0.1412,NaN,NaN


In [58]:
df_agg[df_agg.duplicated(subset=['period', 'symbol'])].sort_values(by=['period', 'symbol'])[['period', 'symbol']]['symbol'].unique()

array([], dtype=object)

In [59]:
df_agg = df_agg.drop_duplicates()
df_agg[df_agg.duplicated(subset=['period', 'symbol'])].sort_values(by=['period', 'symbol']).columns

Index(['accountsPayable', 'accountsReceivables', 'accruedLiability',
       'accumulatedDepreciation', 'additionalPaidInCapital', 'cash',
       'cashEquivalents', 'cashShortTermInvestments', 'commonStock',
       'currentAssets',
       ...
       'otherFundsFinancingItems', 'otherInvestingCashFlowItemsTotal',
       'statement_type', 'cashDividendsPaid', 'changesinWorkingCapital',
       'depreciationAmortization_cf', 'netIncomeStartingLine',
       'otherFundsNonCashItems', 'deferredTaxesInvestmentTaxCredit',
       'stockBasedCompensation'],
      dtype='object', length=112)

In [ ]:
import numpy as np

# Helper: avoid division by zero
def safe_div(n, d):
    return np.where(d == 0, np.nan, n / d)

# Revenue growth (YoY or QoQ depending on your frequency)
df_agg = df_agg.sort_values(by=["symbol", "period"])
df_agg["revenue_growth"] = df_agg.groupby("symbol")["revenue"].pct_change()

# Profitability
df_agg["gross_profit"] = df_agg["grossIncome"]       # already gross profit
df_agg["operating_profit"] = df_agg["ebit"]
df_agg["net_profit"] = df_agg["netIncome"]

# Margins
df_agg["gross_margin"] = safe_div(df_agg["grossIncome"], df_agg["revenue"])
df_agg["operating_margin"] = safe_div(df_agg["ebit"], df_agg["revenue"])
df_agg["net_margin"] = safe_div(df_agg["netIncome"], df_agg["revenue"])

# Liquidity
df_agg["current_ratio"] = safe_div(df_agg["currentAssets"], df_agg["currentLiabilities"])
df_agg["quick_ratio"] = safe_div(df_agg["currentAssets"] - df_agg["inventory"],
                                 df_agg["currentLiabilities"])

# Leverage
df_agg["debt_equity"] = safe_div(df_agg["totalDebt"], df_agg["totalEquity"])

# Returns
df_agg["roa"] = safe_div(df_agg["netIncome"], df_agg["totalAssets"])
df_agg["roe"] = safe_div(df_agg["netIncome"], df_agg["totalEquity"])

df_agg["dupont_net_margin"] = safe_div(df_agg["netIncome"], df_agg["revenue"])
df_agg["dupont_asset_turnover"] = safe_div(df_agg["revenue"], df_agg["totalAssets"])
df_agg["dupont_equity_multiplier"] = safe_div(df_agg["totalAssets"], df_agg["totalEquity"])

df_agg["dupont_roe"] = (
    df_agg["dupont_net_margin"] *
    df_agg["dupont_asset_turnover"] *
    df_agg["dupont_equity_multiplier"]
)

ratios = [
    "revenue_growth", "gross_profit", "operating_profit", "net_profit",
    "gross_margin", "operating_margin", "net_margin",
    "current_ratio", "quick_ratio",
    "roa", "roe", "debt_equity",
    "dupont_net_margin", "dupont_asset_turnover", "dupont_equity_multiplier", "dupont_roe"
]

df_final = df_agg[["period", "symbol"] + ratios]
df_final = df_final.rename(columns={'period': 'Date', 'symbol': 'Ticker'}).sort_values(by=['Date', 'Ticker'])
.reset_index(drop=True)
combined_fs_agg = df_final.copy()
assert len(combined_fs_agg[combined_fs_agg.duplicated(subset=['Date', 'Ticker'])]) == 0

SyntaxError: invalid syntax (3655669459.py, line 53)

: 

In [20]:
combined_fs_agg[combined_fs_agg.duplicated(subset=['Date', 'Ticker'])]

,Date,Ticker,revenue_growth,gross_profit,operating_profit,net_profit,gross_margin,operating_margin,net_margin,current_ratio,quick_ratio,roa,roe,debt_equity,dupont_net_margin,dupont_asset_turnover,dupont_equity_multiplier,dupont_roe
3,1989-03-31,DVN,NaN,4.30,0.500,-0.200,0.597222,0.069444,-0.027778,NaN,NaN,NaN,NaN,NaN,-0.027778,NaN,NaN,NaN
4,1989-03-31,DVN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,1989-03-31,VZ,NaN,1152.30,590.500,330.800,0.416248,0.213308,0.119496,NaN,NaN,NaN,NaN,NaN,0.119496,NaN,NaN,NaN
8,1989-03-31,VZ,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,1989-06-30,AAPL,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163017,2025-08-31,COST,0.363120,11119.00,3341.000,2610.000,0.129057,0.038778,0.030294,NaN,NaN,NaN,NaN,NaN,0.030294,NaN,NaN,NaN
163018,2025-08-31,COST,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
163020,2025-08-31,NIAKF,0.007239,2267.11,582.562,419.108,0.377445,0.096989,0.069776,NaN,NaN,NaN,NaN,NaN,0.069776,NaN,NaN,NaN
163022,2025-08-31,SVLDF,-2.724670,-84.00,-84.000,-82.100,1.072797,1.072797,1.048531,NaN,NaN,NaN,NaN,NaN,1.048531,NaN,NaN,NaN


In [17]:
df_final = df_final.rename(columns={'period': 'Date', 'symbol': 'Ticker'}).sort_values(by=['Date', 'Ticker']).reset_index(drop=True)
combined_fs_agg = df_final.copy()

### Treatment of Historical ESG Scores ###

In [18]:
# We just keep the aggregated esg scores 
historical_esg_agg = historical_esg[['period', 'symbol', 'environmentScore', 'governanceScore','socialScore', 'totalESGScore']].copy().rename(columns={'period': 'Date', 'symbol': 'Ticker'}).sort_values(by=['Date', 'Ticker']).reset_index(drop=True)
historical_esg_agg

,Date,Ticker,environmentScore,governanceScore,socialScore,totalESGScore
0,2010-01-30,MRVL,52.957930,62.350246,41.206560,52.171577
1,2010-01-31,WMT,78.284195,68.111330,76.607445,74.334320
2,2010-02-28,AMKYF,35.558550,47.244200,5.422739,29.408497
3,2010-03-20,JSNSF,78.035130,70.429830,91.688255,80.051070
4,2010-03-31,SFTBF,22.544886,68.871630,23.969582,38.462032
...,...,...,...,...,...,...
311,2024-09-30,IMBBF,99.000000,88.010155,95.145775,94.051980
312,2024-10-31,HURC,8.315232,60.733112,40.691500,36.579950
313,2024-12-26,NCMI,48.668716,23.395193,65.357315,45.807076
314,2024-12-28,KLG,8.242188,36.514423,23.395832,22.717482


### Treatment of Insider Sentiment ###


In [19]:
import pandas as pd

insider_sentiment = pd.read_parquet(f'insider_sentiment.parquet')

# Create Date column as first day of month
insider_sentiment["Date"] = pd.to_datetime(dict(year=insider_sentiment["year"], month=insider_sentiment["month"], day=1))

# Rename mspr
insider_sentiment = insider_sentiment.rename(columns={"mspr": "monthly_share_purchase_ratio", 'change': 'net_change_insider_transactions'}).sort_values(by='Date').drop_duplicates()

# Shift within each symbol group
insider_sentiment["prev_monthly_share_purchase_ratio"] = (
    insider_sentiment.groupby("symbol")["monthly_share_purchase_ratio"].shift(1)
)
insider_sentiment_agg = insider_sentiment[['Date', 'symbol', 'net_change_insider_transactions', 'monthly_share_purchase_ratio', 'prev_monthly_share_purchase_ratio']].rename(columns={'symbol': 'Ticker'}).sort_values(by=['Date', 'Ticker']).reset_index(drop=True)
insider_sentiment_agg


,Date,Ticker,net_change_insider_transactions,monthly_share_purchase_ratio,prev_monthly_share_purchase_ratio
0,2005-01-01,AAPL,-1759517,-33.453964,NaN
1,2005-01-01,COST,-18000,-33.333332,NaN
2,2005-01-01,MSFT,-3177482,-88.085750,NaN
3,2005-01-01,OPK,5000,100.000000,NaN
4,2005-01-01,RGS,45042,21.694023,NaN
...,...,...,...,...,...
16814,2025-09-01,A,-2000,-100.000000,-100.000000
16815,2025-09-01,AARD,26000,100.000000,0.000000
16816,2025-09-01,ADM,-8344,-60.842934,-100.000000
16817,2025-09-01,ADV,144508,100.000000,-3.207173


### Treatment of Insider Transactions ###

In [ ]:
insider_transactions_agg = insider_transactions[['filingDate', 'symbol', 'change', 'share', 'transactionPrice']].copy()
insider_transactions_agg['before_share'] = insider_transactions_agg['share'] - insider_transactions_agg['change']
insider_transactions_agg['change_vol'] = insider_transactions_agg['change'] * insider_transactions_agg['transactionPrice']
insider_transactions_agg = insider_transactions_agg.rename(columns={'symbol': 'Ticker', 'filingDate': 'Date'}).sort_values(by=['Date', 'Ticker']).reset_index(drop=True)
insider_transactions_agg

,Date,Ticker,change,share,transactionPrice,before_share,change_vol
0,,ADMRF,7769083,NaN,0.0070,NaN,54383.581
1,,ADMRF,15927035,NaN,0.0070,NaN,111489.245
2,,ADMRF,10000000,NaN,0.0120,NaN,120000.000
3,,ADMRF,24599,NaN,1.0100,NaN,24844.990
4,,ADMRF,1056000,NaN,0.0330,NaN,34848.000
...,...,...,...,...,...,...,...
398468,2025-09-16,WCUFF,750000,3750000.0,0.0500,4500000.0,37500.000
398469,2025-09-16,WCUFF,750000,3350000.0,0.0500,4100000.0,37500.000
398470,2025-09-16,WGTFF,50000,84433.0,0.1600,134433.0,8000.000
398471,2025-09-16,WMT,-1610,645968.0,103.7100,644358.0,-166973.100


### Treatment of Market Cap History ###

In [21]:
market_cap_agg = market_cap.rename(columns={'symbol': 'Ticker', 'atDate': 'Date'}).sort_values(by=['Date', 'Ticker']).reset_index(drop=True)
market_cap_agg

,Date,marketCapitalization,Ticker
0,2005-09-19,4.109080e+07,PTAIF
1,2005-09-20,4.007870e+07,PTAIF
2,2005-09-21,3.845940e+07,PTAIF
3,2005-09-22,3.663760e+07,PTAIF
4,2005-09-23,3.663760e+07,PTAIF
...,...,...,...
6195,2025-09-12,5.928364e+08,AMMNF
6196,2025-09-14,2.671693e+04,ICL
6197,2025-09-15,5.902900e+08,AMMNF
6198,2025-09-16,5.892105e+08,AMMNF


### Treatment of Social Sentiment ###

In [22]:
social_sentiment['Date'] = pd.to_datetime(social_sentiment['atTime']).dt.date
social_sentiment_agg = social_sentiment[['Date', 'symbol', 'score']].rename(columns={'symbol': 'Ticker', 'score': 'social_sentiment_score'}).sort_values(by=['Date', 'Ticker']).reset_index(drop=True)
social_sentiment_agg

,Date,Ticker,social_sentiment_score
0,2021-03-15,ADMA,0.6000
1,2021-03-15,CBIO,-0.6000
2,2021-03-15,GOGO,-0.6000
3,2021-03-15,KOD,0.0000
4,2021-03-15,NEOV,0.0000
...,...,...,...
26242,2025-09-16,WMT,-0.0022
26243,2025-09-17,A,-0.8649
26244,2025-09-17,MSFT,-0.6000
26245,2025-09-17,SO,0.0758


### Treatment of USA Spending History ###

In [23]:
usa_spending_history_agg = usa_spending_history[['actionDate', 'symbol', 'totalValue']]
usa_spending_history_agg = usa_spending_history_agg[usa_spending_history_agg['totalValue'] > 0].rename(columns={'symbol': 'Ticker', 'actionDate': 'Date'}).sort_values(by=['Date', 'Ticker']).reset_index(drop=True)
usa_spending_history_agg

,Date,Ticker,totalValue
0,2007-10-01,ATUS,6.193200e+02
1,2007-10-01,ATUS,2.722400e+04
2,2007-10-01,BANF,6.484800e+02
3,2007-10-01,BANF,6.484800e+02
4,2007-10-01,BANF,6.484800e+02
...,...,...,...
206696,2025-09-04,VZ,3.742412e+08
206697,2025-09-04,VZ,2.627111e+04
206698,2025-09-04,VZ,1.000000e+03
206699,2025-09-04,VZ,5.436160e+03
